In [ ]:
import os
import re
from unidecode import unidecode

In [ ]:
import pandas as pd

# Função para carregar contagens de palavras (por ano) num único DataFrame
def carrega_palavras(anos, prefixo='data/result/words'):
    frames = []
    for ano in anos:
        df = pd.read_csv(f"{prefixo}{ano}.csv")
        frames.append(df[['palavra','frequencia']].assign(ano=ano))
    return pd.concat(frames)

# Carrega dados
ref_df = carrega_palavras([2020, 2021, 2022])
alvo_df = carrega_palavras([2023, 2024, 2025])

# Soma frequência por palavra em cada período
ref_group = ref_df.groupby('palavra')['frequencia'].sum()
alvo_group = alvo_df.groupby('palavra')['frequencia'].sum()

# Soma total de palavras em cada período
total_ref = ref_group.sum()
total_alvo = alvo_group.sum()

# Index de todas as palavras observadas em qualquer período
all_words = set(ref_group.index).union(alvo_group.index)

resultados = []

for palavra in all_words:
    freq_ref = ref_group.get(palavra, 0)
    freq_alvo = alvo_group.get(palavra, 0)
    # Se a palavra não ocorre no período de referência, usa pseudocontagem 1
    freq_ref_adj = freq_ref if freq_ref > 0 else 1

    # Razão de excesso
    r = (freq_alvo / total_alvo) / (freq_ref_adj / total_ref)
    resultados.append({'palavra': palavra, 'freq_alvo': freq_alvo, 'freq_ref': freq_ref, 'r': r})

# Cria DataFrame, ordena por maior razão de excesso
df_r = pd.DataFrame(resultados)
df_r = df_r.sort_values('r', ascending=False)

# Para visualizar os 20 maiores excessos:
print(df_r.head(20))

# Salva resultado em CSV
df_r.to_csv('data/razao_excesso_2023_2025_vs_2020_2022.csv', index=False)
print("Arquivo salvo em data/razao_excesso_2023_2025_vs_2020_2022.csv")

In [6]:
import pandas as pd

# refinar: excluir dados numéricos, selecionar razão de excesso (r>2) e frequencia alvo >100

# Caminho do arquivo original
entrada = "data/razao_excesso_2023_2025_vs_2020_2022.csv"

# Nome do novo arquivo
saida = "data/razao_excesso_2023_2025_vs_2020_2022_freqalvo100.csv"

# Ler o CSV
df = pd.read_csv(entrada)

# 1) Excluir linhas com números na coluna "palavras"
df = df[~df["palavra"].astype(str).str.contains(r"\d", regex=True)]

# 2) Excluir linhas com razão (coluna "r") menor que 2
df = df[df["r"] >= 2]

# 3) Excluir linhas com freq_alvo menor que 100
df = df[df["freq_alvo"] >= 100]

# Salvar o novo arquivo
df.to_csv(saida, index=False)

print(f"Arquivo salvo com sucesso em: {saida}")
print(f"Total de linhas após filtragem: {len(df)}")


Arquivo salvo com sucesso em: data/razao_excesso_2023_2025_vs_2020_2022_freqalvo100.csv
Total de linhas após filtragem: 1478


In [2]:
import pandas as pd

# Carregue seu arquivo CSV
df = pd.read_csv('data/razao_excesso_2023_2025_vs_2020_2022_freqalvo100.csv')

In [2]:
import pandas as pd

# Carregar a planilha original
df = pd.read_csv('data/razao_excesso_2023_2025_vs_2020_2022_freqalvo100_formatado.csv')

# Lista de palavras de estilo selecionadas manualmente para filtrar
palavras_alvo = [
    'pactual', 'perfilado', 'notarial', 'assistida', 'checagem', 'afastadas', 'indumentaria', 'experiencial',
    'afinidade', 'atipica', 'deliberada', 'notariais', 'pioneiro', 'opcional', 'recuos', 'desestabilizacao',
    'duvidosa', 'agradabilidade', 'incidencias', 'depositadas', 'persistentes', 'insolvencia', 'comparabilidade',
    'interseccionais', 'uniformizacao', 'crucial', 'interseccional', 'atrativos', 'intensivos', 'expositivo',
    'alinhadas', 'agravo', 'equitativa', 'alinhando', 'equitativo', 'arcabouco', 'situacional', 'insurgencia',
    'robusta', 'justifique', 'envolvente', 'desenvolvimentismo', 'quantilica', 'efetuar', 'permeabilidade',
    'autocorrelacao', 'cruciais', 'disparidades', 'excepcionalidade', 'robustas', 'estacionarias', 'turbulencia',
    'moderador', 'associativas', 'robustos', 'edificados', 'desempenhou', 'contextualizada', 'caminham', 'esboco',
    'mobilizadas', 'discursivas', 'esperadas', 'discursiva', 'transcende', 'promovendo', 'assegurando', 'efetuados',
    'argumentou', 'difusao', 'similaridade', 'recomendado', 'globalmente', 'exploram', 'ressaltou', 'desempenha',
    'instrumentalizacao', 'desempenham', 'interseccionalidade', 'enfrenta', 'enfrentam', 'colaborando', 'eficazes',
    'impulsionando', 'desafia', 'intersecao', 'refletindo', 'pesquisado', 'transformador', 'enfrentados', 'estudando',
    'enunciacao', 'revelou', 'inovadoras', 'alinhamento', 'explicativas', 'enfatizou', 'intrinsecamente', 'expostas',
    'enfatizando', 'explicativa', 'divulga', 'alinhar', 'perpetua', 'desafiar', 'robusto', 'moldando', 'impulsionada',
    'destacou', 'fortalece', 'ponderada', 'abordou', 'destacando', 'inspirar', 'enfrentou', 'inovadora',
    'proporcionando', 'problematizacao', 'garantindo', 'indicador', 'desafiador', 'engajados', 'frequentemente',
    'identificamos', 'sugerindo', 'rejeita', 'fortalecendo', 'condutor', 'convergencias', 'visualizacao',
    'acionaveis', 'recuadas', 'opcionais', 'caminhabilidade', 'eficiencias'
]

# Filtrar o DataFrame pelas palavras da lista (coluna 'palavra')
df_filtrado = df[df['palavra'].isin(palavras_alvo)]

# Salvar o resultado em uma nova planilha
df_filtrado.to_csv('data/razao_excesso_2023_2025_vs_2020_2022_freqalvo100_filtrado.csv', index=False)

In [15]:
import pandas as pd

# Ajuste o caminho se necessário
caminho_csv = "corpus_geral_ano_PPG_link.csv"
# caminho_csv = "EBBC/corpus_geral_ano_PPG_link.csv"

# Leitura do arquivo
df = pd.read_csv(caminho_csv)

# Ajuste o nome da coluna se necessário
df['ano'] = df['ano'].astype(int)

# Contagem por ano
quantidade_por_ano = df.groupby('ano').size().reset_index(name='quantidade')

# Ordenar por ano
quantidade_por_ano = quantidade_por_ano.sort_values('ano')

# Exibir lista
print(quantidade_por_ano)




    ano  quantidade
0  2020         142
1  2021         153
2  2022         202
3  2023         175
4  2024         170
5  2025          75


In [ ]:
import os
import unicodedata
from collections import defaultdict

def remover_acentos(texto):
    return ''.join(
        c for c in unicodedata.normalize('NFKD', texto)
        if not unicodedata.combining(c)
    )

# 1. Lê e normaliza as palavras
with open('data/razao_excesso_2023_2025_vs_2020_2022_freqalvo100_filtrado.csv', 'r', encoding='utf-8') as f:
    palavras = [linha.strip() for linha in f if linha.strip()]
palavras_normalizadas = [remover_acentos(palavra.lower()) for palavra in palavras]

# 2. Define as pastas e arquivos
pastas = [
    'data2020/txt2020',
    'data2021/txt2021',
    'data2022/txt2022'
]
arquivos = []
for pasta in pastas:
    for nome_arquivo in os.listdir(pasta):
        if nome_arquivo.endswith('.txt'):
            arquivos.append(os.path.join(pasta, nome_arquivo))

# 3. Conta em quantos documentos cada palavra aparece
contagem = defaultdict(int)

for arquivo in arquivos:
    try:
        with open(arquivo, 'r', encoding='utf-8') as f:
            conteudo = f.read().lower()
            conteudo_normalizado = remover_acentos(conteudo)
            for idx, palavra_normalizada in enumerate(palavras_normalizadas):
                if palavra_normalizada in conteudo_normalizado:
                    contagem[palavras[idx]] += 1
    except Exception as e:
        print(f'Erro ao ler {arquivo}: {e}')

# 4. Exibe o resultado
for palavra in palavras:
    print(f'{palavra}: {contagem[palavra]}')

In [ ]:
import os
import unicodedata
from collections import defaultdict

def remover_acentos(texto):
    return ''.join(
        c for c in unicodedata.normalize('NFKD', texto)
        if not unicodedata.combining(c)
    )

# 1. Lê e normaliza as palavras
with open('data/razao_excesso_2023_2025_vs_2020_2022_freqalvo100_filtrado.csv', 'r', encoding='utf-8') as f:
    palavras = [linha.strip() for linha in f if linha.strip()]
palavras_normalizadas = [remover_acentos(palavra.lower()) for palavra in palavras]

# 2. Define as pastas e arquivos
pastas = [
    'data2023/txt2023',
    'data2024/txt2024',
    'data2025/txt2025'
]
arquivos = []
for pasta in pastas:
    for nome_arquivo in os.listdir(pasta):
        if nome_arquivo.endswith('.txt'):
            arquivos.append(os.path.join(pasta, nome_arquivo))

# 3. Conta em quantos documentos cada palavra aparece
contagem = defaultdict(int)

for arquivo in arquivos:
    try:
        with open(arquivo, 'r', encoding='utf-8') as f:
            conteudo = f.read().lower()
            conteudo_normalizado = remover_acentos(conteudo)
            for idx, palavra_normalizada in enumerate(palavras_normalizadas):
                if palavra_normalizada in conteudo_normalizado:
                    contagem[palavras[idx]] += 1
    except Exception as e:
        print(f'Erro ao ler {arquivo}: {e}')

# 4. Exibe o resultado
for palavra in palavras:
    print(f'{palavra}: {contagem[palavra]}')